In [ ]:
# ── Starter Cell 1: Load environment variables ──────────────────────────────
from dotenv import load_dotenv   # imports the dotenv helper that reads .env files
import os                         # stdlib module for environment variable access

load_dotenv()                     # reads .env in the current directory and sets env vars
api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the key from the environment
print("API key loaded:", api_key is not None)  # True means the key was found

In [ ]:
# ── Starter Cell 2: Initialise the Anthropic client ─────────────────────────
from anthropic import Anthropic   # the official Python SDK class

client = Anthropic(api_key=api_key)  # creates an authenticated client instance
print("Anthropic client ready")       # confirms object was constructed without error

In [ ]:
# ── Starter Cell 3: Smoke test — verify end-to-end connectivity ──────────────
response = client.messages.create(
    model="claude-haiku-4-5",          # fast, cheap model — good for connectivity checks
    #    model="claude-sonnet-4-5",
    max_tokens=50,                      # hard cap; smoke test only needs a short reply
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
    # messages is a list of dicts — the minimum required field besides model/max_tokens
)
print(response.content[0].text)        # content is a list of blocks; [0] is the first text block

# CCA Lab: Structured Data Exercise — Prefill + Stop Sequence

**Course:** Building with the Claude API  
**Sub-module:** Accessing Claude with the API  
**Lesson:** Structured Data Exercise

## What this lab covers
- Assistant message prefilling to lock output format before generation begins
- `stop_sequences` to terminate generation at a known boundary
- Combining prefill + stop sequences for clean, fence-free structured extraction
- Applying the pattern to real-world AWS CLI command generation
- Full improvement cycle: write → evaluate → improve → re-evaluate
- Failure mode analysis and token usage tracking across all API calls

## CCA Domains Exercised
- **Primary:** Prompt Engineering
- **Contributing:** Agentic Architecture (API parameter mechanics)

## Learning Objectives
1. Explain *why* prefilling is more reliable than asking Claude to "only return code" in prose.
2. Demonstrate the canonical prefill + stop sequence pattern for structured extraction.
3. Identify which characters are *excluded* from the response when a stop sequence fires.
4. Score prompt quality with a rubric and improve iteratively.
5. Recognise and handle at least three failure modes specific to this pattern.

## Section 1: API Glossary and Parameter Reference
### CCA objective demonstrated: Know every parameter involved in the prefill + stop sequence pattern and what is excluded from the returned output when a stop sequence fires.

### API Parameter Reference — `stop_sequences` and Assistant Prefill

| Parameter | Type | Required | Purpose |
|---|---|---|---|
| `model` | `str` | ✅ Yes | Selects which Claude model processes the request. |
| `max_tokens` | `int` | ✅ Yes | Hard upper bound on generated tokens; prevents runaway output. |
| `messages` | `list[dict]` | ✅ Yes | Alternating `user`/`assistant` turn list that provides conversation context. |
| `messages[-1]` with `role="assistant"` | `dict` | For prefill | An assistant entry at the *end* of `messages` tells Claude it has already started that reply — it continues from there instead of starting fresh. |
| `stop_sequences` | `list[str]` | ❌ Optional | List of strings that, when generated, immediately halt output. The stop string itself is **not** included in the returned text. |
| `system` | `str` | ❌ Optional | Top-level instruction that frames Claude's role before any user turn. |

**What is excluded when a stop sequence fires?**

| Scenario | Included in `response.content[0].text`? |
|---|---|
| Text generated *before* the stop sequence | ✅ Yes |
| The stop sequence string itself (e.g. ` ``` `) | ❌ No — excluded |
| Text that would have followed the stop sequence | ❌ No — never generated |

> **Architectural principle:** The stop sequence is a *fence post*, not a *fence panel* — it marks the boundary but is never part of the returned content.

## Section 2: ask() Helper — Full Statement-by-Statement Annotation
### CCA objective demonstrated: Build a reusable helper that supports system prompts, assistant prefilling, stop sequences, and usage logging.

In [ ]:
# ── ask() helper + conversation utilities ────────────────────────────────────

usage_log = []   # module-level list; every API call appends a usage record here

def ask(
    messages,                  # the full conversation list (user + optional assistant turns)
    system="",                 # system prompt string — empty string means no system prompt
    model="claude-haiku-4-5", # default model; override for higher-quality tasks
    max_tokens=512,            # generous default; structured extraction usually needs less
    stop_sequences=None,       # list of strings that halt generation — None means no stops
    label="call",              # human-readable tag stored in usage_log for reporting
):
    """
    Send a messages list to Claude and return the assistant's text.

    Parameters
    ----------
    messages      : list of {role, content} dicts — the conversation so far
    system        : optional top-level instruction string
    model         : Claude model ID string
    max_tokens    : hard token ceiling for the response
    stop_sequences: list of strings that terminate generation; stop string excluded from output
    label         : tag for the usage_log entry, useful for per-call reporting

    Returns
    -------
    str — the assistant's reply text (stop sequence itself is NOT included)
    """
    kwargs = {                 # build keyword-argument dict dynamically
        "model": model,        # required — identifies which Claude model to use
        "max_tokens": max_tokens,  # required — API rejects calls without this
        "messages": messages,  # required — the conversation turns
    }
    if system:                 # only add 'system' key when a non-empty string is provided
        kwargs["system"] = system  # system is a top-level param, NOT inside messages[]

    if stop_sequences:         # None or empty list → skip; truthy list → add to request
        kwargs["stop_sequences"] = stop_sequences  # API accepts a list of stop strings

    response = client.messages.create(**kwargs)  # unpack dict as keyword args; makes call

    usage_log.append({         # record token usage AFTER each call so we can report totals
        "label": label,                              # human-readable call name
        "input_tokens": response.usage.input_tokens, # tokens sent (prompt + context)
        "output_tokens": response.usage.output_tokens,# tokens generated by Claude
        "stop_reason": response.stop_reason,         # 'end_turn', 'stop_sequence', 'max_tokens'
    })

    # response.content is a list because Claude can return mixed content blocks
    # (text blocks, tool_use blocks, etc.). We access [0] for the first — and in
    # non-tool calls the only — text block.
    return response.content[0].text if response.content else ""   # return just the string, not the full response object


def add_user_message(messages, text):
    """Append a user turn to the messages list in-place and return the list."""
    messages.append({"role": "user", "content": text})   # role must be exactly 'user'
    return messages   # returning allows chaining, though mutation already happened


def add_assistant_message(messages, text):
    """
    Inject an assistant prefill turn.

    When the LAST entry in messages has role='assistant', Claude treats that
    text as something it already wrote and continues the response from that
    point — this is the prefilling mechanism.
    """
    messages.append({"role": "assistant", "content": text})  # role must be exactly 'assistant'
    return messages   # return for chaining


print("ask(), add_user_message(), add_assistant_message() defined.")  # confirm definitions loaded

## Section 3: Core Concept — Prefill + Stop Sequence Structured Extraction
### CCA objective demonstrated: Apply the prefill + stop_sequences pattern to extract clean bash commands from Claude with no fences, no prose, and no post-processing.

In [ ]:
# ── D-STRUCT: Prefill + Stop Sequence — Live Extraction Demo ─────────────────
#
# CANONICAL PATTERN:
#   1. Build messages list with a user request
#   2. Inject assistant prefill that OPENS the code fence
#   3. Call ask() with stop_sequences=["```"] to halt at the CLOSING fence
#   4. Returned text = clean content between the opened fence and the stop
#
# WHY THIS WORKS:
#   Claude sees the assistant turn as something it already wrote, so it
#   continues writing bash code — it never decides to add prose or wrap
#   in markdown because the fence is already open.
# ─────────────────────────────────────────────────────────────────────────────

messages = []   # fresh conversation list — start empty for each independent extraction

add_user_message(
    messages,
    # User turn: ask for exactly three very short AWS CLI commands.
    # 'very short' keeps tokens low and makes the output easy to inspect.
    "Give me 3 very short sample AWS CLI commands that list resources."
)

# ── PREFILL TURN ─────────────────────────────────────────────────────────────
# The prefill string tells Claude it has ALREADY begun its response with this
# exact text. The \n after ```bash is important — it mirrors real fence syntax
# so Claude's continuation aligns with standard bash fencing.
prefill_text = "Here are all three commands in a single block without any comments:\n```bash"
add_assistant_message(messages, prefill_text)  # inject the prefill as the last turn

# ── API CALL WITH STOP SEQUENCE ───────────────────────────────────────────────
# stop_sequences=["```"] tells Claude: when you generate those three backticks,
# stop immediately and do NOT include them in the output.
# Result: we get bash lines only — no opening fence (stripped by prefill logic),
# no closing fence (halted by stop sequence), no prose.
raw_output = ask(
    messages,
    stop_sequences=["```"],   # halt when the closing fence would be generated
    label="prefill-stop-demo" # tag for usage_log
)

print("=== RAW RETURNED TEXT (no fences, no prose) ===")
print(raw_output)   # print exactly what the API returned — should be pure bash lines

print("\n=== VERIFICATION ===")
# Check that no markdown fence characters leaked into the output.
# re.search scans the whole string (not just the start like re.match),
# \b word-boundary anchors prevent matching ``` inside longer strings.
import re   # standard library regular expression module
has_fence = bool(re.search(r'`{3}', raw_output))   # True if three backticks found anywhere
has_prose = bool(re.search(r'^(here|these|below|i\s)', raw_output.strip().lower()))  # prose openers

print(f"Contains markdown fence (```): {has_fence}  — expected False")
print(f"Starts with prose opener     : {has_prose}  — expected False")

# Show the stop_reason so we can confirm the stop sequence fired as intended.
print(f"stop_reason                  : {usage_log[-1]['stop_reason']}  — expected stop_sequence")

## Section 4: System Prompt — Role and Decision Framework
### CCA objective demonstrated: Distinguish what belongs in the system parameter versus the user turn, and apply system prompts alongside prefilling.

### System Parameter Decision Table

| Put it in **`system`** | Put it in the **user turn** |
|---|---|
| Persistent role or persona ("You are an AWS expert.") | The specific task for this request |
| Output format rules that never change across turns | Dynamic parameters (region, service name) |
| Safety constraints and tone guidelines | Context that varies per call |
| Domain knowledge injected for every call | One-off instructions specific to this prompt |

**Architectural principle:** The `system` parameter is a *standing order* that frames every turn; the user turn is the *mission brief* for a single request.

> **Note for this lesson:** The prefill + stop sequence pattern can work with *or* without a system prompt. When combined, the system prompt constrains Claude's persona/domain, while the prefill constrains its output *format*.

In [ ]:
# ── System Prompt + Prefill: Combined Pattern ────────────────────────────────
# Here we add a system prompt that scopes Claude to AWS CLI topics,
# then still use prefill + stop to lock the output format.

messages2 = []   # new messages list for this demonstration call

add_user_message(
    messages2,
    "Give me 3 very short AWS CLI commands that describe EC2 instances."
    # user turn: specific task — EC2 describe commands
)

add_assistant_message(
    messages2,
    "```bash"   # shorter prefill — we rely on system prompt for prose suppression
    # The prefill still opens the code fence; system prompt handles the 'no comments' rule
)

system_with_prefill = (
    "You are an AWS CLI expert. "                         # role — goes in system
    "Return only valid AWS CLI commands, no prose, "      # format rule — goes in system
    "no inline comments, no markdown outside code fences."  # persistent constraint
)

output2 = ask(
    messages2,
    system=system_with_prefill,   # system prompt scopes role + format globally
    stop_sequences=["```"],       # stop sequence still handles fence boundary
    label="system-plus-prefill"   # label for usage_log
)

print("=== OUTPUT WITH SYSTEM PROMPT + PREFILL ===")
print(output2)
print(f"\nstop_reason: {usage_log[-1]['stop_reason']}")

## Section 5: Multi-Turn Conversation — Messages List Accumulation
### CCA objective demonstrated: Show how the messages list grows turn-by-turn, and how prefill turns appear inside that list alongside real user/assistant turns.

In [ ]:
# ── Multi-Turn with Prefill: Watching the Messages List Grow ─────────────────
# This cell builds a two-turn conversation and prints the FULL messages list
# after each turn so the reader can see every role/content entry accumulate.

import json   # for pretty-printing the messages list as indented JSON

mt_messages = []   # fresh list for this multi-turn demonstration

# ── TURN 1 ───────────────────────────────────────────────────────────────────
add_user_message(mt_messages, "List 2 very short AWS S3 CLI commands.")  # first user ask
add_assistant_message(mt_messages, "```bash")  # prefill opens the fence

print("=== messages list BEFORE Turn 1 API call ===")
# Use json.dumps for readable indented output; default=str handles any non-serialisable values
print(json.dumps(mt_messages, indent=2, default=str))

reply1 = ask(
    mt_messages,
    stop_sequences=["```"],   # halt at closing fence
    label="multi-turn-1"      # usage log label
)
# After the API call, replace the prefill entry with the full assistant reply
# so the conversation history is accurate for the next turn.
mt_messages[-1]["content"] = "```bash\n" + reply1 + "```"   # reconstruct the full fenced block

print("\n=== messages list AFTER Turn 1 (assistant entry updated with full reply) ===")
print(json.dumps(mt_messages, indent=2, default=str))

# ── TURN 2 ───────────────────────────────────────────────────────────────────
add_user_message(mt_messages, "Now give me 2 very short AWS IAM CLI commands.")  # second ask
add_assistant_message(mt_messages, "```bash")   # prefill for turn 2

print("\n=== messages list BEFORE Turn 2 API call (4 entries now) ===")
print(json.dumps(mt_messages, indent=2, default=str))   # list has grown to 4 entries

reply2 = ask(
    mt_messages,
    stop_sequences=["```"],
    label="multi-turn-2"
)
mt_messages[-1]["content"] = "```bash\n" + reply2 + "```"  # update with full reply

print("\n=== messages list AFTER Turn 2 (final state, 4 entries) ===")
print(json.dumps(mt_messages, indent=2, default=str))

print("\n=== EXTRACTED COMMANDS ===")
print("S3 commands:\n", reply1)
print("IAM commands:\n", reply2)

## Section 6: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
### CCA objective demonstrated: Score prompt quality with a numeric rubric, iterate across three versions, and confirm V3 clears the pass threshold.

> **Grader reliability note:** Deterministic rubrics can over-score responses that contain the right words but wrong semantics. For production evals, complement keyword checks with a Claude-as-judge semantic pass — the rule + judge layering pattern from the evaluation labs.

In [ ]:
# ── Rubric Definition ─────────────────────────────────────────────────────────
# Design philosophy (Rules A–D):
#   - Every dimension has a direct lever in the V3 system/prefill prompt
#   - V1 (vague prompt) predictably fails at least 2 dimensions
#   - V2 fixes exactly 1 of those failures
#   - V3 fixes ALL failures; score meets PASS_THRESHOLD
#   - PASS_THRESHOLD is set so only V3 crosses it
# ─────────────────────────────────────────────────────────────────────────────

PASS_THRESHOLD = 4   # only V3 should reach this; V1 and V2 must both score below it

def score_response(text, label=""):
    """
    Score a Claude response on four rubric dimensions (1 point each, max 4).

    Dimensions:
      D1 — No markdown fences in the returned text (stop sequence worked)
      D2 — No prose openers (no 'Here are', 'These commands', etc.)
      D3 — Starts with 'aws ' (action verb = CLI command, not a heading or comment)
      D4 — Contains exactly 3 command lines (one per line, no blank lines padding)

    Parameters
    ----------
    text  : str — the raw text returned by Claude
    label : str — identifier printed alongside scores

    Returns
    -------
    int — total score 0–4
    """
    # ── Rule A: heading-aware first-sentence extraction ───────────────────────
    # text.split('.')[0] or text[:N] would fail when Claude adds a heading.
    # We skip any line starting with '#' to find the true first prose line.
    first_line = next(
        (s.strip() for s in text.split('\n')    # iterate over lines
         if s.strip() and not s.strip().startswith('#')),  # skip headings and blank lines
        ''   # default if no qualifying line found
    )

    # D1: No markdown fences anywhere in the output
    # re.search (not re.match) scans the whole string, not just the start
    d1 = int(not bool(re.search(r'`{3}', text)))   # int(bool()) converts True/False → 1/0
    print(f"  {'✅' if d1 else '❌'} D1 no-fence         : {d1}/1")

    # D2: First line does not begin with a prose opener
    # \b word boundary ensures we match 'here' as a word, not 'here_after'
    d2 = int(not bool(re.search(r'^(here|these|below|the following)', first_line.lower())))
    print(f"  {'✅' if d2 else '❌'} D2 no-prose-opener  : {d2}/1")

    # D3: First line starts with 'aws ' — confirms it's a CLI command, not a comment
    d3 = int(first_line.lower().startswith('aws '))   # 'aws ' with trailing space avoids 'awesome'
    print(f"  {'✅' if d3 else '❌'} D3 starts-with-aws  : {d3}/1")

    # D4: Exactly 3 non-empty lines (one command per line)
    # List comprehension: keep lines that are non-empty after stripping whitespace
    non_empty_lines = [ln for ln in text.split('\n') if ln.strip()]  # filter blank lines
    d4 = int(len(non_empty_lines) == 3)   # int(bool) → 1 if exactly 3, else 0
    print(f"  {'✅' if d4 else '❌'} D4 exactly-3-lines  : {d4}/1  (found {len(non_empty_lines)})")

    total = d1 + d2 + d3 + d4   # sum of four binary dimensions
    print(f"  ── {label} TOTAL: {total}/{PASS_THRESHOLD}")
    return total


print("score_response() defined. PASS_THRESHOLD =", PASS_THRESHOLD)

In [ ]:
# ── Improvement Cycle: V1 → V2 → V3 ─────────────────────────────────────────
#
# V1 — Vague prompt: no prefill, no stop sequence, no format constraint
#       Expected failures: D1 (fences present), D2 (prose opener), D3 (no 'aws ' first)
# V2 — Adds stop sequence only (no prefill): fixes D1, still may fail D2 and D3
# V3 — Full pattern: prefill + stop sequence + system prompt forcing aws-first start
#       Expected to pass all four dimensions
# ─────────────────────────────────────────────────────────────────────────────

# ── V1: Vague — no prefill, no stop sequence ─────────────────────────────────
v1_messages = [{"role": "user", "content": "Give me 3 AWS CLI commands."}]
# No prefill, no stop_sequences — Claude will likely add prose and fences
v1_output = ask(v1_messages, label="v1-vague")
print("\n=== V1 OUTPUT ===")
print(repr(v1_output[:200]))   # repr() shows \n characters explicitly
print("--- V1 SCORE ---")
v1_score = score_response(v1_output, "V1")

# ── V2: Adds stop sequence but NO prefill ────────────────────────────────────
# Without a prefill, the stop sequence alone doesn't prevent prose before the fence.
# V2 fixes D1 (no fence in output) but likely still fails D2 and D3.
v2_messages = [{"role": "user", "content": "Give me 3 AWS CLI commands in a bash block."}]
# No prefill — stop sequence will strip the fence but prose may still appear
v2_output = ask(v2_messages, stop_sequences=["```\n"], label="v2-stop-only")
print("\n=== V2 OUTPUT ===")
print(repr(v2_output[:200]))
print("--- V2 SCORE ---")
v2_score = score_response(v2_output, "V2")

# ── V3: Full pattern — system prompt + prefill + stop sequence ────────────────
# System prompt: 'Do not use markdown headings. Start with an action verb (aws)."
# Prefill: opens the fence so Claude continues with bash commands immediately
# Stop sequence: halts at closing fence
# This directly addresses every dimension V1 and V2 failed.
v3_system = (
    "You are an AWS CLI expert. "
    "Do not use markdown headings. "
    "Do not start with prose. "
    "Start your very first line with an aws command (begin with 'aws '). "
    "No inline comments. Output exactly 3 commands, one per line."
)
v3_messages = [{"role": "user", "content": "Give me 3 very short AWS CLI commands."}]
add_assistant_message(v3_messages, "```bash")   # prefill opens the fence
v3_output = ask(
    v3_messages,
    system=v3_system,          # system scopes role + no-heading + aws-first rules
    stop_sequences=["```"],    # halt at closing fence
    label="v3-full-pattern"
)
print("\n=== V3 OUTPUT ===")
print(repr(v3_output[:200]))
print("--- V3 SCORE ---")
v3_score = score_response(v3_output, "V3")

# ── Side-by-side comparison table ────────────────────────────────────────────
print("\n" + "=" * 62)
print(f"{'Version':<10} {'Prompt (truncated)':<32} {'Score':>6}")
print("-" * 62)

# Extract truncated prompt text outside the f-string to avoid backslash-in-fstring SyntaxError
v1_snippet = v1_messages[0]['content'][:28]
v2_snippet = v2_messages[0]['content'][:28]
v3_snippet = v3_messages[0]['content'][:28]

print(f"{'V1 (vague)':<10} {v1_snippet:<32} {v1_score:>4}/{PASS_THRESHOLD}")
print(f"{'V2 (stop)':<10} {v2_snippet:<32} {v2_score:>4}/{PASS_THRESHOLD}")
print(f"{'V3 (full)':<10} {v3_snippet:<32} {v3_score:>4}/{PASS_THRESHOLD}")
print("=" * 62)

# Conditional PASS/FAIL line based on threshold
result = "✅ PASS" if v3_score >= PASS_THRESHOLD else "❌ FAIL"
print(f"V3 result: {result} (score {v3_score} vs threshold {PASS_THRESHOLD})")

## Section 7: Failure Mode Analysis
### CCA objective demonstrated: Identify, describe, and trigger failure modes specific to the prefill + stop sequence pattern.

### Failure Mode Reference Table

| Failure Mode | Trigger | Symptom |
|---|---|---|
| **Missing stop sequence** | `stop_sequences` omitted or empty | Response includes the closing ` ``` ` fence and any prose Claude writes after it — post-processing needed. |
| **Prefill mismatch** | Prefill opens wrong fence type (e.g. ` ```python ` for bash) | Claude may switch languages mid-response or add a corrective comment, contaminating the output. |
| **max_tokens too low** | `max_tokens` hit before stop sequence fires | `stop_reason = 'max_tokens'` instead of `'stop_sequence'`; output is truncated mid-command. |
| **No prefill, stop only** | Assistant prefill turn omitted | Claude writes prose introduction before the fence; the prose leaks into the returned text since stop fires only at the closing fence. |
| **Stop sequence in content** | A command legitimately contains ` ``` ` (rare) | Generation halts early inside a command line, producing an incomplete output. |

In [ ]:
# ── Live Failure Mode Demo ────────────────────────────────────────────────────
# We deliberately trigger Failure Mode 3: max_tokens too low.
# Setting max_tokens=5 guarantees the model cannot generate even one full command
# before hitting the token ceiling, producing a truncated output.

failure_messages = []
add_user_message(failure_messages, "Give me 3 very short AWS CLI commands.")  # standard ask
add_assistant_message(failure_messages, "```bash")  # standard prefill

# ── INTENTIONAL FAILURE: max_tokens deliberately set to 5 ────────────────────
failure_output = ask(
    failure_messages,
    stop_sequences=["```"],   # stop sequence present but will never fire — hits token limit first
    max_tokens=5,             # ← too low: triggers max_tokens stop reason, not stop_sequence
    label="failure-max-tokens"
)

# Retrieve the stop_reason from the usage log
failure_stop_reason = usage_log[-1]['stop_reason']  # should be 'max_tokens'

print("=== FAILURE DEMO OUTPUT ===")
print(repr(failure_output))   # repr shows truncation clearly

print("\n=== FAILURE ANALYSIS ===")
# Guard rail: check stop_reason and raise a descriptive error if truncation occurred
if failure_stop_reason == "max_tokens":
    # Raise a RuntimeError with a clear message explaining the failure
    try:
        raise RuntimeError(
            f"[Failure Mode 3] Output truncated: stop_reason='{failure_stop_reason}'. "
            f"The stop sequence never fired because max_tokens ({5}) was exhausted first. "
            "Increase max_tokens or reduce prompt complexity to allow the stop sequence to fire."
        )
    except RuntimeError as e:
        # Catch and print — we want the notebook to continue running for the reader
        print(f"❌ RuntimeError caught: {e}")
else:
    print(f"stop_reason was '{failure_stop_reason}' — no max_tokens failure detected.")

# Also demonstrate Failure Mode 1: missing stop sequence leaks the closing fence
print("\n=== FAILURE MODE 1 DEMO: missing stop_sequences ===")
leak_messages = []
add_user_message(leak_messages, "Give me 1 very short AWS CLI command.")  # short to save tokens
add_assistant_message(leak_messages, "```bash")
leak_output = ask(
    leak_messages,
    stop_sequences=None,   # ← deliberately omitted — closing fence will appear in output
    max_tokens=80,
    label="failure-no-stop"
)
has_closing_fence = bool(re.search(r'`{3}', leak_output))   # check for leaked fence
print(f"Output: {repr(leak_output[:150])}")
print(f"Closing fence leaked into output: {has_closing_fence}  — demonstrates the failure")

## Section 8: Token Usage Tracking
### CCA objective demonstrated: Report per-call token usage with input, output, and cumulative totals across all API calls made in this lab.

In [ ]:
# ── Token Usage Report ────────────────────────────────────────────────────────
# usage_log was populated by every ask() call throughout this notebook.
# We now print a per-call table with input_tokens, output_tokens, and
# running cumulative totals so the reader can see cost accumulation.

print(f"{'#':<3} {'Label':<28} {'Input':>7} {'Output':>7} {'CumIn':>8} {'CumOut':>8} {'Stop Reason':<16}")
print("-" * 85)

cumulative_input = 0    # running total of input tokens
cumulative_output = 0   # running total of output tokens

for idx, entry in enumerate(usage_log, start=1):   # enumerate from 1 for human-readable index
    cumulative_input  += entry['input_tokens']    # accumulate input tokens
    cumulative_output += entry['output_tokens']   # accumulate output tokens

    # Extract values into variables before f-string to avoid backslash-in-expression error
    lbl   = entry['label'][:26]           # truncate long labels to fit column
    inp   = entry['input_tokens']         # input tokens for this call
    out   = entry['output_tokens']        # output tokens for this call
    stop  = entry['stop_reason']          # stop_reason string

    print(f"{idx:<3} {lbl:<28} {inp:>7} {out:>7} {cumulative_input:>8} {cumulative_output:>8} {stop:<16}")

print("-" * 85)
print(f"{'TOTAL':<32} {cumulative_input:>7} {cumulative_output:>7}  (across {len(usage_log)} calls)")
print(f"\nGrand total tokens used this session: {cumulative_input + cumulative_output}")

## Section 9: Practice Drills
### CCA objective demonstrated: Apply the prefill + stop sequence pattern independently to new structured extraction tasks.

In [ ]:
# ── Drill 1: JSON Extraction with Prefill + Stop ──────────────────────────────
# Task: Use the prefill + stop sequence pattern to extract a clean JSON object
# containing three AWS services with their short descriptions — no prose, no fences.
#
# PATTERN: prefill with '{' (opening brace) → stop_sequences=['}'] (closing brace)
# The returned text is the JSON interior WITHOUT the surrounding braces.

drill1_messages = []   # fresh messages list
add_user_message(
    drill1_messages,
    "Give me a JSON object with 3 keys: s3, ec2, lambda. "
    "Each value is a 5-word description. No other text."
)
add_assistant_message(drill1_messages, "{")   # prefill opens the JSON object

drill1_output = ask(
    drill1_messages,
    stop_sequences=["}"],   # halt at the closing brace — brace excluded from output
    max_tokens=150,
    label="drill1-json"
)

# Reconstruct the full JSON by wrapping the output in braces
full_json_str = "{" + drill1_output + "}"   # the prefill '{' + content + the stop '}'
print("Reconstructed JSON string:")
print(full_json_str)

# Validate that the reconstructed string is parseable JSON
try:
    parsed = json.loads(full_json_str)   # attempt to parse; raises ValueError on failure
    print("\n✅ Valid JSON — parsed keys:", list(parsed.keys()))
except json.JSONDecodeError as e:
    print(f"\n❌ JSON parse error: {e}")   # report parse failure with details

In [ ]:
# ── Drill 2: CSV Extraction ───────────────────────────────────────────────────
# Task: Extract a clean 3-row CSV (no header) of AWS regions using prefill + stop.
# PATTERN: prefill with first row → stop_sequences=["\n\n"] (double newline = end of block)

drill2_system = (
    "Output only CSV data. No headers, no prose, no markdown. "
    "Each row: region_code,continent,typical_latency_ms"
)
drill2_messages = []
add_user_message(
    drill2_messages,
    "Give me 3 AWS regions as CSV rows: region_code, continent, typical_latency_ms."
)
# Prefill injects the first row so Claude continues with rows 2 and 3
add_assistant_message(drill2_messages, "us-east-1,NorthAmerica,10")

drill2_output = ask(
    drill2_messages,
    system=drill2_system,
    stop_sequences=["\n\n"],   # double newline signals end of CSV block
    max_tokens=100,
    label="drill2-csv"
)

# Reconstruct full CSV by prepending the prefilled first row
full_csv = "us-east-1,NorthAmerica,10\n" + drill2_output   # prefill row + generated rows
print("=== EXTRACTED CSV ===")
print(full_csv)

# Verify: split into lines and check count
csv_lines = [ln for ln in full_csv.split('\n') if ln.strip()]   # filter blank lines
print(f"\nRow count: {len(csv_lines)}  (expected 3)")
for i, row in enumerate(csv_lines, 1):
    cols = row.split(',')   # split each CSV row by comma
    print(f"  Row {i}: {len(cols)} columns — {row[:60]}")

In [ ]:
# ── Drill 3: Multi-Service Command Extraction ─────────────────────────────────
# Task: Use separate prefill + stop calls for two AWS services (S3 and EC2)
# and store the results in a dict keyed by service name.
# This demonstrates reusability of the pattern across multiple extraction calls.

def extract_aws_commands(service_name, count=2):
    """
    Extract clean bash AWS CLI commands for a given service using prefill + stop.

    Parameters
    ----------
    service_name : str — AWS service name (e.g. 's3', 'ec2')
    count        : int — number of commands to request

    Returns
    -------
    str — raw bash commands, one per line, no fences, no prose
    """
    msgs = []   # fresh messages list per extraction
    add_user_message(
        msgs,
        f"Give me exactly {count} very short AWS {service_name.upper()} CLI commands."
        # f-string interpolates service name and count into the request
    )
    add_assistant_message(msgs, "```bash")   # standard prefill opens the fence
    result = ask(
        msgs,
        system=(
            f"You are an AWS {service_name.upper()} CLI expert. "
            "Do not use headings. Start each line with 'aws '. No comments."
        ),
        stop_sequences=["```"],    # halt at closing fence
        max_tokens=120,
        label=f"drill3-{service_name}"   # label includes service for usage_log clarity
    )
    return result   # return the clean command string


services = ["s3", "ec2", "iam"]   # list of services to extract commands for
command_registry = {}              # dict to store results keyed by service name

for svc in services:               # iterate over each service
    command_registry[svc] = extract_aws_commands(svc, count=2)  # extract and store
    print(f"\n=== {svc.upper()} COMMANDS ===")
    print(command_registry[svc])   # print the extracted commands for this service

print("\n=== REGISTRY SUMMARY ===")
# Count non-empty lines per service to verify extraction
for svc, cmds in command_registry.items():   # .items() yields (key, value) pairs
    line_count = len([ln for ln in cmds.split('\n') if ln.strip()])  # count non-empty lines
    print(f"  {svc.upper()}: {line_count} command(s) extracted")

## Section 10: CCA Takeaways — 5 Exam-Ready Mental Models
### CCA objective demonstrated: Consolidate the lesson into portable, exam-applicable rules.

| # | Mental Model | Why It Matters for the CCA Exam |
|---|---|---|
| 1 | **Prefill = context injection, not instruction** — An assistant turn at the end of `messages[]` makes Claude *believe* it already started responding. It continues, not interprets. | Exam Q: "How do you force Claude to begin a response with a specific token?" → Prefill the assistant turn, not a system instruction. |
| 2 | **Stop sequence boundary is exclusive** — The stop string itself is never included in `response.content[0].text`. You must re-attach it if you need it in the final output. | Exam Q: "What does Claude return when a stop sequence fires?" → Text *before* the stop sequence; the stop string is excluded. |
| 3 | **stop_reason is your guard rail** — Always check `response.stop_reason`. `'stop_sequence'` = clean extraction; `'max_tokens'` = truncation; `'end_turn'` = no stop fired. | Exam Q: "How do you detect truncation in a structured extraction call?" → Check `stop_reason == 'max_tokens'`. |
| 4 | **Prefill + stop > prose instruction for format control** — Telling Claude "only return code" in prose fails stochastically. Opening the code fence in the prefill is deterministic. | Exam Q: "What is more reliable for extracting clean bash output?" → Prefill the fence opening, not a prose constraint. |
| 5 | **The pattern generalises across all delimited formats** — JSON (`{` → `}`), CSV (row 1 → `\n\n`), SQL (`SELECT` → `;`), bash (` ```bash\n ` → ` ``` `). Same mechanism, different delimiters. | Exam Q: "Apply the prefill + stop technique to JSON extraction." → Prefill `{`, stop on `}`, reconstruct by wrapping. |